# BGE-M3 semantic chunking → Qdrant

Run the cells in order. The pipeline reads **To read Data** at the repository root (including subfolders), calls the backend's existing `chunk_documents`, embeds the resulting chunks with **BAAI/bge-m3**, and upserts dense vectors into Qdrant.

Supported inputs: UTF-8 **CSV files only**. Set `TEXT_COLUMN` to the CSV column containing preprocessed text; other columns are retained as metadata. Missing/empty folders skip ingestion before loading the model or connecting to Qdrant. Unsupported files are reported.

Start Qdrant separately, or set `QDRANT_URL` and optionally `QDRANT_API_KEY` in the notebook kernel environment for your server. The default is `http://localhost:6333`. The first real ingestion downloads the embedding model.


In [ ]:
%pip install -q langchain-experimental==0.4.2 sentence-transformers qdrant-client


In [ ]:
import csv
import os
import sys
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

# Locate the repository when launched from its root, backend, or notebook folder.
PROJECT_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / "lucraftmeds")
     if (candidate / "lucraftmeds@be" / "chunking" / "chunking.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from within the LucraftMeds workspace.")
sys.path.insert(0, str(PROJECT_ROOT / "lucraftmeds@be"))

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from chunking.chunking import chunk_documents
from qdrant_client import QdrantClient, models

DATA_DIR = PROJECT_ROOT / "To read Data"  # Change to your folder if needed.
TEXT_COLUMN = "text"
QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")
COLLECTION_NAME = os.environ.get("QDRANT_COLLECTION", "lucraftmeds_bge_m3")
MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 32
MAX_TOKENS = 512
BREAKPOINT_PERCENTILE = 95

print(f"Input folder: {DATA_DIR}")
print(f"Qdrant collection: {COLLECTION_NAME}")


In [ ]:
def iter_documents(data_dir, text_column=TEXT_COLUMN):
    """Yield nonblank documents, retaining CSV metadata and source identity."""
    data_dir = Path(data_dir)
    if not data_dir.exists():
        print(f"Folder not found: {data_dir}. Skipping ingestion.")
        return
    if not data_dir.is_dir():
        raise NotADirectoryError(data_dir)

    for path in sorted(data_dir.rglob("*")):
        if not path.is_file():
            continue
        source = path.relative_to(data_dir).as_posix()
        if path.suffix.lower() == ".csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle, strict=True)
                if text_column not in (reader.fieldnames or []):
                    raise ValueError(f"{source}: missing text column {text_column!r}.")
                for row_number, row in enumerate(reader, start=1):
                    if None in row or any(value is None for value in row.values()):
                        raise ValueError(f"{source}: malformed CSV row {row_number}.")
                    text = row.pop(text_column)
                    if text.strip():
                        yield Document(
                            page_content=text,
                            metadata={**row, "source_file": source, "source_row": row_number},
                        )
        else:
            print(f"Skipping non-CSV file: {source}")


In [ ]:
class BGEM3Embeddings(Embeddings):
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(MODEL_NAME)

    def embed_documents(self, texts):
        return self.model.encode(
            texts, batch_size=BATCH_SIZE, normalize_embeddings=True,
            show_progress_bar=False,
        ).tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]

    def count_tokens(self, text):
        return len(self.model.tokenizer.encode(text, add_special_tokens=True))


In [ ]:
def embed_and_upload_chunks(client, collection_name, chunks, embeddings):
    """Embed chunk text with BGE-M3 before uploading vectors to Qdrant."""
    uploaded = 0
    for offset in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[offset:offset + BATCH_SIZE]
        vectors = embeddings.embed_documents([chunk.page_content for chunk in batch])
        if len(vectors) != len(batch) or not vectors or not vectors[0]:
            raise ValueError("Embedding model returned an invalid number of vectors.")
        dimension = len(vectors[0])
        if any(len(vector) != dimension for vector in vectors):
            raise ValueError("Embedding dimensions are inconsistent.")
        if offset == 0:
            if not client.collection_exists(collection_name):
                client.create_collection(
                    collection_name,
                    vectors_config=models.VectorParams(size=dimension, distance=models.Distance.COSINE),
                )
            else:
                config = client.get_collection(collection_name).config.params.vectors
                if (not isinstance(config, models.VectorParams)
                        or config.size != dimension or config.distance != models.Distance.COSINE):
                    raise ValueError(
                        f"Collection {collection_name!r} needs unnamed {dimension}-dimensional "
                        "cosine vectors. Choose a compatible collection name."
                    )
        points = []
        for chunk, vector in zip(batch, vectors):
            # Source location + content makes unchanged reruns idempotent.
            identity = repr((
                MODEL_NAME, chunk.metadata["source_file"], chunk.metadata.get("source_row"),
                chunk.metadata.get("start_index"), chunk.page_content,
            ))
            points.append(models.PointStruct(
                id=str(uuid5(NAMESPACE_URL, identity)),
                vector=vector,
                payload={"page_content": chunk.page_content, "metadata": chunk.metadata,
                         "embedding_model": MODEL_NAME},
            ))
        client.upsert(collection_name=collection_name, points=points, wait=True)
        uploaded += len(points)
    return uploaded


In [ ]:
def ingest_folder():
    documents = iter_documents(DATA_DIR)
    first = next(documents, None)
    if first is None:
        print("No nonblank CSV rows to ingest.")
        return {"documents": 0, "chunks": 0}

    embeddings = BGEM3Embeddings()
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)
    document_count = chunk_count = 0
    try:
        document = first
        while document is not None:
            chunks = chunk_documents(
                [document], embeddings,
                breakpoint_threshold_amount=BREAKPOINT_PERCENTILE,
                max_tokens=MAX_TOKENS, token_count=embeddings.count_tokens,
            )
            chunk_count += embed_and_upload_chunks(client, COLLECTION_NAME, chunks, embeddings)
            document_count += 1
            print(f"Uploaded {chunk_count:,} chunks from {document_count:,} documents.")
            document = next(documents, None)
    finally:
        client.close()
        documents.close()
    return {"documents": document_count, "chunks": chunk_count}


## Run ingestion

Existing collections are reused without deletion. Use a collection dedicated to this embedding model. Upserts follow the [Qdrant client API](https://qdrant.tech/documentation/quickstart/); embeddings use [SentenceTransformer.encode](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html).

Unchanged input has stable point IDs. **Edited or removed source documents can leave older points in the collection**; use a fresh collection for a clean rebuild. If a file fails during processing, earlier successful uploads remain and the error is raised. Correct the input and rerun.


In [ ]:
ingestion_result = ingest_folder()
print(ingestion_result)
